In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
CONNECTION_STRING = "sqlite:///C:/ProgramData/studiocyfrowe_apps/Lumos/LumosDatabase.db"

In [6]:
engine = create_engine(CONNECTION_STRING)
sql_processes = """SELECT 
        StationName, 
        TotalGB, 
        UsedGB, 
        FreeGB, 
        UsedPercent, 
        LastScan
    FROM MemoryRamScans
    ORDER BY LastScan DESC
"""

In [7]:
data_df = pd.read_sql(sql_processes, CONNECTION_STRING)
data_df.head()

,StationName,TotalGB,UsedGB,FreeGB,UsedPercent,LastScan
0,DESKTOP-RJ92I5V,15.72,12.84,2.88,81.7,2026-03-29T13:36:32.8730156Z
1,DESKTOP-RJ92I5V,15.72,12.99,2.73,82.6,2026-03-29T13:35:53.0171031Z
2,DESKTOP-RJ92I5V,15.72,12.90,2.82,82.1,2026-03-29T13:35:13.0483342Z
3,DESKTOP-RJ92I5V,15.72,12.86,2.86,81.8,2026-03-29T13:34:32.5299940Z
4,DESKTOP-RJ92I5V,15.72,12.99,2.73,82.6,2026-03-29T13:33:51.9871720Z


In [8]:
grouped = data_df.groupby('StationName').agg({
    "UsedPercent": ['mean', 'max']
})

In [10]:
grouped = grouped.reset_index()
grouped.columns = ['StationName', 'used_percent_mean', 'used_percent_max']

In [11]:
grouped.head()

,StationName,used_percent_mean,used_percent_max
0,DESKTOP-RJ92I5V,81.371667,96.4


In [12]:
issues = []

if grouped["used_percent_mean"].mean() > 80:
    issues.append("Sustained high CPU usage across processes")

if grouped["used_percent_max"].max() > 80:
    issues.append("Single process reached very high CPU usage (possible CPU spike)")

In [16]:
payload = {
    "processor_scan": {
        "processor_used_percent_mean": round(grouped["used_percent_mean"].mean(), 2),
        "processor_used_percent_max": round(grouped["used_percent_max"].max(), 2)
    },
    "issues": issues
}

In [17]:
payload

{'processor_scan': {'processor_used_percent_mean': np.float64(81.37),
  'processor_used_percent_max': np.float64(96.4)},
 'issues': ['Sustained high CPU usage across processes',
  'Single process reached very high CPU usage (possible CPU spike)']}